# 01 — Medium-range forecast from ERA5 initial conditions

This notebook runs a **10-day (240 h) autoregressive forecast** with the
XiChen state-forecast model, starting from an ERA5 initial condition, and
scores it against ERA5 (latitude-weighted RMSE / ACC) — the experiment of
Fig. 3 of the XiChen paper.

**Prerequisites** (see README § Download):

- `../ckpts/xichen_state_forecast_ar15.ckpt`
- `../data/era5/` (1.0° npy archive + `normalized_mean_std/` + `climatology_np181x360_2010_2021/`)

Runtime: ~2–5 min on a single GPU for one init time.

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from xichen.data import VARIABLES
from xichen.device import get_device
from xichen.forecast_eval import eval_forecast, make_lr_loader
from xichen import nwp

# --- paths (demo data / ckpts downloaded from Zenodo, see README) ---
DATA_DIR = Path("/fs6/home/yangjh_data/project_data/xichen")
ERA5_DIR = Path("/fs6/home/yangjh_data/project_data/xichen/observation/ERA5")
CKPT_DIR = Path("/fs6/home/yangjh15/xichen/ckpt/xichen_1p0deg")
OUTPUT_DIR = Path("/fs6/home/yangjh15/xichen/XiChen_1p0deg_public/outputs/era5_init_forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- forecast parameters ---
INIT_TIME = datetime(2023, 1, 5, 0)   # demo ERA5 archive starts here
FORECAST_HOURS = 240                  # 10 days
DT = 6                                # 6-hourly evaluation
device = get_device("cuda", 0)
print("device:", device)


In [ ]:
# Run the AR rollout + evaluation (writes NetCDF fields, CSVs and figures).
config = {
    "era5_lr_dir": str(ERA5_DIR),
    "era5_hr_dir": str(ERA5_DIR),   # unused at 1.0°, placeholder
    "forecast_config": "../configs/xichen_forecast.json",
    "forecast_hours": FORECAST_HOURS,
    "dt": DT,
    "forecast_name": "xichen_era5init_demo",
    "device": device,
    "forecast_pair": "lr",
    "resolution_tag": "1p0deg",
    "output_resolution": "1p0",
    "init_times": [INIT_TIME],
    "eval_batch_size": 1,
}
metrics = eval_forecast(
    make_lr_loader(str(ERA5_DIR)),
    str(CKPT_DIR / "xichen_state_forecast_ar15.ckpt"),
    config,
    str(OUTPUT_DIR),
)
print("RMSE shape (n_vars, n_leads):", metrics["rmse"].shape)

In [ ]:
# Latitude-weighted RMSE / ACC vs lead time for key variables.
leads = np.arange(0, FORECAST_HOURS + 1, DT)
plot_vars = ["z-500", "t-850", "t2m", "u10"]
idx = [VARIABLES.index(v) for v in plot_vars]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for v, i in zip(plot_vars, idx):
    axes[0].plot(leads, metrics["rmse"][i], marker="o", ms=3, label=v)
    axes[1].plot(leads, metrics["acc"][i], marker="o", ms=3, label=v)
axes[0].set(title="RMSE (lat-weighted)", xlabel="lead time [h]")
axes[0].set_ylim(bottom=0)
axes[1].set(title="ACC (anomaly correlation)", xlabel="lead time [h]")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

# Persist for the comparison in notebook 03.
np.savez(
    OUTPUT_DIR / "era5_init_metrics.npz",
    rmse=metrics["rmse"], acc=metrics["acc"], leads=leads, variables=VARIABLES,
)
print("saved:", OUTPUT_DIR / "era5_init_metrics.npz")

In [ ]:
from dateutil.relativedelta import relativedelta
# Forecast field vs ERA5 truth vs error at t+24h / t+120h / t+240h (z-500).
Z500 = VARIABLES.index("z-500")
loader = make_lr_loader(str(ERA5_DIR))

fig, axes = plt.subplots(3, 3, figsize=(15, 9), constrained_layout=True)
for row, lead in enumerate([24, 120, 240]):
    fc = nwp.load_field(nwp.field_path(OUTPUT_DIR / "forecast", INIT_TIME, lead))
    truth, _ = loader(INIT_TIME + relativedelta(hours=lead))
    fc_z, tr_z = fc[Z500], truth[0, Z500]
    err = fc_z - tr_z
    vmin, vmax = np.nanmin(tr_z), np.nanmax(tr_z)
    emax = np.nanmax(np.abs(err))
    for col, (data, title, cmap, lims) in enumerate([
        (tr_z, f"ERA5 z-500", "viridis", (vmin, vmax)),
        (fc_z, f"XiChen z-500 t+{lead}h", "viridis", (vmin, vmax)),
        (err, f"error t+{lead}h", "RdBu_r", (-emax, emax)),
    ]):
        im = axes[row, col].imshow(data, cmap=cmap, vmin=lims[0], vmax=lims[1])
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
        plt.colorbar(im, ax=axes[row, col], shrink=0.8)
plt.show()

**Outputs** (all under `../outputs/era5_init_forecast/`):

- `forecast/<init>/<date>-<lead>.nc` — forecast fields (NWP-Benchmark layout)
- `xichen_era5init_demo_1p0deg_{rmse,acc,activity,pred_rmse}.csv` — metrics
- `era5_init_metrics.npz` — consumed by notebook **03** for the DA-init comparison

Next: `02_dacycle.ipynb` — build an analysis by assimilating observations.